## Diagnostic checks are performed for the following models

### Model 1. Current conflict intensity
$$
\log(rent_{itk}) = \alpha + \mu_i + \lambda_t + \delta_k + \beta_1 total\_attacks_{it} + \gamma_1 fatalities_{it} + \gamma_2 unemp\_monthly_{it} + \varepsilon_{itk}
$$

### Model 2. Lagged attacks
$$
\log(rent_{itk}) = \alpha + \mu_i + \lambda_t + \delta_k + \beta_1 total\_attacks\_lag1_{it} + \gamma_1 fatalities_{it} + \gamma_2 unemp\_monthly_{it} + \varepsilon_{itk}
$$

### Model 3. Attack composition
$$
\log(rent_{itk}) = \alpha + \mu_i + \lambda_t + \delta_k + \beta_1 shelling\_count_{it} + \beta_2 drone\_count_{it} + \beta_3 battle\_count_{it} + \gamma_1 fatalities_{it} + \gamma_2 unemp\_monthly_{it} + \varepsilon_{itk}
$$

### Model 4. Attack dummy
$$
\log(rent_{itk}) = \alpha + \mu_i + \lambda_t + \delta_k + \beta_1 any\_attack_{it} + \gamma_1 fatalities_{it} + \gamma_2 unemp\_monthly_{it} + \varepsilon_{itk}
$$

In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from IPython.display import display

df = pd.read_csv("rent_acled_with_lags.csv")

df = df.rename(columns={
    "область": "region",
    "тип_квартири": "flat_type",
    "середня_ціна_грн": "rent_uah"
})
df["region"] = df["region"].astype(str)
df["month"] = df["month"].astype(str)
df["flat_type"] = df["flat_type"].astype(str)

def fit_fe_model_plain(data, y, x_vars):
    needed_cols = [y, "region", "month", "flat_type"] + x_vars
    d = data[needed_cols].dropna().copy()

    formula = y + " ~ " + " + ".join(x_vars) + " + C(region) + C(month) + C(flat_type)"
    model = smf.ols(formula=formula, data=d).fit()
    return model, d

def breusch_pagan_test(model, data, x_vars):
    X = add_constant(data[x_vars], has_constant="add").astype(float)
    lm_stat, lm_pvalue, f_stat, f_pvalue = het_breuschpagan(model.resid, X)

    return pd.DataFrame({
        "LM_stat": [lm_stat],
        "LM_pvalue": [lm_pvalue],
        "F_stat": [f_stat],
        "F_pvalue": [f_pvalue]
    })

def vif_table(data, x_vars):
    X = add_constant(data[x_vars], has_constant="add").astype(float)

    rows = []
    for i, col in enumerate(X.columns):
        if col == "const":
            continue
        rows.append({
            "variable": col,
            "VIF": variance_inflation_factor(X.values, i)
        })

    return pd.DataFrame(rows).sort_values("VIF", ascending=False).reset_index(drop=True)

In [4]:
diagnostic_specs = {
    "Model 1: current conflict intensity": ["total_attacks", "fatalities", "unemp_monthly"],
    "Model 2: lagged attacks": ["total_attacks_lag1", "fatalities", "unemp_monthly"],
    "Model 3: attack composition": ["shelling_count", "drone_count", "battle_count", "fatalities", "unemp_monthly"],
    "Model 4: attack dummy": ["any_attack", "fatalities", "unemp_monthly"]
}

for model_name, x_vars in diagnostic_specs.items():
    model, d = fit_fe_model_plain(df, y="log_rent", x_vars=x_vars)

    bp_res = breusch_pagan_test(model, d, x_vars)
    vif_res = vif_table(d, x_vars)

    print("\n" + "=" * 100)
    print(model_name)

    print("\nBreusch-Pagan test")
    display(bp_res)

    print("\nVIF")
    display(vif_res)


Model 1: current conflict intensity

Breusch-Pagan test


,LM_stat,LM_pvalue,F_stat,F_pvalue
0,53.462784,1.460752e-11,18.201605,1.124616e-11



VIF


,variable,VIF
0,total_attacks,1.497953
1,fatalities,1.424153
2,unemp_monthly,1.075330



Model 2: lagged attacks

Breusch-Pagan test


...

,variable,VIF
0,shelling_count,1.717437
1,fatalities,1.700722
2,battle_count,1.543752
3,drone_count,1.319442
4,unemp_monthly,1.165923



Model 4: attack dummy

Breusch-Pagan test


,LM_stat,LM_pvalue,F_stat,F_pvalue
0,31.437928,6.874103e-07,10.602314,6.349389e-07



VIF


,variable,VIF
0,any_attack,1.130380
1,unemp_monthly,1.095023
2,fatalities,1.035880
